# Three model multimodal fusion

This notebook combines the **text classifier**, **ANN numeric classifier**, and **CNN thumbnail classifier** by averaging their clickbait probabilities (soft voting). It uses the existing train, validation, and test splits and applies each model's own preprocessing.

The text model and CNN load their trained checkpoints. The ANN notebook saves its fitted model and training-fitted scaler to `models/ann_model.joblib`; fusion loads that artifact and runs inference directly. The ANN is not retrained here.

Run from the repository root or `fusion/`. The text checkpoint is tracked with Git LFS, so make sure Git LFS has fetched the real file (not the small pointer text) before running this notebook.

## 1. Imports, paths, and checkpoints

In [6]:
from pathlib import Path
import re
import subprocess
import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, log_loss
from transformers import AutoModel, AutoTokenizer

SEED = 42
LABEL = "final_label"
ID = "video_id"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16 if DEVICE.type == "cuda" else 4
torch.manual_seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").is_dir() and (ROOT.parent / "data" / "processed").is_dir():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "processed"
TEXT_DIR = ROOT / "models" / "text_classifier"
TEXT_CHECKPOINT = TEXT_DIR / "best_model.pt"
CNN_CHECKPOINT = ROOT / "models" / "resnet18_best.pt"
OUTPUT_DIR = ROOT / "artifacts" / "fusion" / "predictions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in (TEXT_CHECKPOINT, CNN_CHECKPOINT):
    if not path.is_file():
        raise FileNotFoundError(f"Required trained checkpoint not found: {path}")

def is_lfs_pointer(path):
    with Path(path).open("rb") as checkpoint_file:
        return checkpoint_file.read(120).startswith(b"version https://git-lfs.github.com/spec/v1")

if is_lfs_pointer(TEXT_CHECKPOINT):
    print("Fetching the text classifier weights from Git LFS...")
    try:
        pull = subprocess.run(
            ["git", "lfs", "pull", "--include=models/text_classifier/best_model.pt"],
            cwd=ROOT,
            capture_output=True,
            text=True,
            check=False,
        )
    except FileNotFoundError as exc:
        raise RuntimeError(
            "Git LFS is not installed or Git is unavailable. Install Git LFS, run 'git lfs install', "
            "then run 'git lfs pull' from the repository folder."
        ) from exc
    if pull.returncode != 0:
        raise RuntimeError(
            "Could not fetch the text weights with Git LFS. Check Git LFS installation, GitHub access, "
            "and that the LFS object was uploaded, then run 'git lfs pull' from the repository folder."
        )
    if is_lfs_pointer(TEXT_CHECKPOINT):
        raise RuntimeError(
            "Git LFS completed but the file is still a pointer. The large model object may not have been "
            "uploaded to GitHub LFS; check the repository's LFS objects."
        )

print("Device:", DEVICE)
print("Text checkpoint:", TEXT_CHECKPOINT)
print("CNN checkpoint:", CNN_CHECKPOINT)

Device: cuda
Text checkpoint: d:\DC\multimodal-clickbait-detection-fresh\models\text_classifier\best_model.pt
CNN checkpoint: d:\DC\multimodal-clickbait-detection-fresh\models\resnet18_best.pt


## 2. Load and validate the existing splits

In [7]:
def load_split(name):
    frame = pd.read_csv(DATA_DIR / f"{name}.csv", dtype={ID: "string"})
    if LABEL not in frame.columns and "label" in frame.columns:
        frame = frame.rename(columns={"label": LABEL})
    required = {ID, LABEL, "title", "description", "local_thumbnail", "views", "likes", "duration"}
    missing = required - set(frame.columns)
    if missing:
        raise KeyError(f"{name}.csv is missing columns required by the models: {sorted(missing)}")
    frame = frame.copy()
    frame[ID] = frame[ID].astype("string").str.strip()
    frame[LABEL] = pd.to_numeric(frame[LABEL], errors="raise").astype(int)
    if frame[ID].isna().any() or frame[ID].eq("").any() or frame[ID].duplicated().any():
        raise ValueError(f"{name} has empty or duplicate video IDs")
    if not set(frame[LABEL].unique()).issubset({0, 1}):
        raise ValueError(f"{name} labels must be 0 or 1")
    frame["title"] = frame["title"].fillna("").astype(str)
    frame["description"] = frame["description"].fillna("").astype(str)
    return frame

splits = {name: load_split(name) for name in ("train", "validation", "test")}
for a, b in (("train", "validation"), ("train", "test"), ("validation", "test")):
    overlap = set(splits[a][ID]) & set(splits[b][ID])
    if overlap:
        raise ValueError(f"{a}/{b} overlap; check the existing split files")
print({name: len(frame) for name, frame in splits.items()})

{'train': 2957, 'validation': 634, 'test': 634}


## 3. Load the fitted ANN artifact

The ANN notebook creates this file after training. Fusion loads the model and scaler from it, then runs ANN inference on the split data.

In [8]:
ANN_CHECKPOINT = ROOT / "models" / "ann_model.joblib"
if not ANN_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Fitted ANN artifact not found: {ANN_CHECKPOINT}. Run models/ann.ipynb through its save cell once."
    )
ann_artifact = joblib.load(ANN_CHECKPOINT)
ann_model = ann_artifact["model"]
ann_scaler = ann_artifact["scaler"]
ANN_FEATURES = ann_artifact["features"]
if ANN_FEATURES != ["views", "likes", "duration_seconds", "like_ratio"]:
    raise ValueError(f"Unexpected ANN feature order in {ANN_CHECKPOINT}: {ANN_FEATURES}")
print(f"Loaded ANN artifact selected at epoch {ann_artifact.get('best_epoch')}")

Loaded ANN artifact selected at epoch 299


## 4. Load text classifier and CNN checkpoints

In [9]:
# These definitions match models/text_classifier.ipynb.
from transformers import AutoModel, AutoTokenizer
TEXT_MODEL_NAME = "distilbert/distilbert-base-cased"
MAX_LENGTH = 256
URL_RE = re.compile(r"(?i)\\b(?:https?://|www\\.)\\S+")
def clean_description(value):
    return URL_RE.sub("[URL]", str(value or ""))

class TextClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(TEXT_MODEL_NAME)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.encoder.config.dim, 2)
    def forward(self, input_ids, attention_mask):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        return self.classifier(self.dropout(hidden[:, 0]))

tokenizer_dir = TEXT_DIR / "tokenizer"
if not tokenizer_dir.is_dir():
    raise FileNotFoundError(f"Saved tokenizer not found: {tokenizer_dir}")
tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, local_files_only=True, use_fast=True)
text_model = TextClassifier().to(DEVICE)
text_checkpoint = torch.load(TEXT_CHECKPOINT, map_location=DEVICE, weights_only=True)
text_model.load_state_dict(text_checkpoint["model_state_dict"])
text_model.eval()

# Match the ResNet18 architecture and 224px ImageNet normalization from models/cnn.ipynb.
class CNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = resnet18(weights=None)
        self.backbone.fc = nn.Sequential(nn.Dropout(p=0.30), nn.Linear(512, 1))
    def forward(self, images):
        return self.backbone(images).squeeze(1)

cnn_model = CNNClassifier().to(DEVICE)
cnn_checkpoint = torch.load(CNN_CHECKPOINT, map_location=DEVICE, weights_only=True)
cnn_model.load_state_dict(cnn_checkpoint["model_state_dict"])
cnn_model.eval()
cnn_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
print("Loaded text and CNN model weights.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded text and CNN model weights.


## 4. Get all three model probabilities and align by video ID

In [10]:
def duration_to_seconds(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    match = re.fullmatch(r"(\d+(?:\.\d+)?)\s*s?", str(value).strip().lower())
    return float(match.group(1)) if match else np.nan

def prepare_ann_features(frame):
    data = frame.copy()
    data["views"] = pd.to_numeric(data["views"], errors="coerce")
    data["likes"] = pd.to_numeric(data["likes"], errors="coerce")
    data["duration_seconds"] = data["duration"].map(duration_to_seconds)
    data["like_ratio"] = data["likes"].div(data["views"].replace(0, np.nan)).fillna(0.0)
    data = data.replace([np.inf, -np.inf], np.nan)
    data = data.dropna(subset=["views", "likes", "duration_seconds"]).copy()
    if (data[["views", "likes", "duration_seconds"]] < 0).any().any():
        raise ValueError("Views, likes, and duration must be non-negative for log1p.")
    data["views"] = np.log1p(data["views"])
    data["likes"] = np.log1p(data["likes"])
    data["duration_seconds"] = np.log1p(data["duration_seconds"])
    return data

class FusionDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        encoded = tokenizer(text=str(row["title"]), text_pair=clean_description(row["description"]),
                            max_length=MAX_LENGTH, truncation="only_second", padding="max_length",
                            return_attention_mask=True, return_tensors="pt")
        image_path = Path(str(row["local_thumbnail"]))
        if not image_path.is_absolute():
            candidates = (ROOT / image_path, ROOT / "Thumbnail" / image_path.name)
            image_path = next((p for p in candidates if p.is_file()), candidates[0])
        if not image_path.is_file():
            raise FileNotFoundError(f"Thumbnail not found: {image_path}")
        with Image.open(image_path) as image:
            image_tensor = cnn_transform(image.convert("RGB"))
        return str(row[ID]), {k: v.squeeze(0) for k, v in encoded.items()}, image_tensor, int(row[LABEL])

def model_probabilities(frame):
    loader = DataLoader(FusionDataset(frame), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    ids, text_probs, cnn_probs, labels = [], [], [], []
    text_model.eval(); cnn_model.eval()
    with torch.inference_mode():
        for batch_ids, text_batch, images, batch_labels in loader:
            text_batch = {k: v.to(DEVICE) for k, v in text_batch.items() if k in {"input_ids", "attention_mask"}}
            images = images.to(DEVICE)
            text_logits = text_model(**text_batch).float()
            cnn_logits = cnn_model(images).float()
            ids.extend(batch_ids)
            text_probs.extend(text_logits.softmax(1)[:, 1].cpu().numpy())
            cnn_probs.extend(cnn_logits.sigmoid().cpu().numpy())
            labels.extend(batch_labels.numpy())
    ann_data = prepare_ann_features(frame)
    ann_x = ann_scaler.transform(ann_data[ANN_FEATURES])
    ann_probs = ann_model.predict_proba(ann_x)[:, 1]
    ann_scores = pd.DataFrame({
        ID: ann_data[ID].astype("string").to_numpy(),
        "ann_probability": ann_probs,
    })
    neural_scores = pd.DataFrame({
        ID: pd.Series(ids, dtype="string"),
        "text_probability": text_probs,
        "cnn_probability": cnn_probs,
        LABEL: labels,
    })
    scores = neural_scores.merge(ann_scores, on=ID, how="inner", validate="one_to_one")
    if len(scores) != len(frame):
        raise ValueError("ANN inputs were invalid for some rows; align complete records across all modalities first.")
    return scores

scores = {name: model_probabilities(frame) for name, frame in splits.items()}
print({name: len(frame) for name, frame in scores.items()})

{'train': 2957, 'validation': 634, 'test': 634}


## 5. Soft-vote fusion and validation review

In [11]:
MODEL_COLUMNS = ["text_probability", "ann_probability", "cnn_probability"]
def add_fusion_probability(frame):
    result = frame.copy()
    result["clickbait_probability"] = result[MODEL_COLUMNS].mean(axis=1)
    result["final_prediction"] = (result["clickbait_probability"] >= 0.5).astype(int)
    return result

validation_fusion = add_fusion_probability(scores["validation"])
val_y = validation_fusion[LABEL].to_numpy()
val_pred = validation_fusion["final_prediction"].to_numpy()
print(f"Validation accuracy: {accuracy_score(val_y, val_pred):.4f}")
print(f"Validation F1: {f1_score(val_y, val_pred, zero_division=0):.4f}")
print(classification_report(val_y, val_pred, labels=[0, 1],
                            target_names=["not clickbait", "clickbait"], zero_division=0))
display(pd.DataFrame(confusion_matrix(val_y, val_pred, labels=[0, 1]),
                     index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"]))

Validation accuracy: 0.9653
Validation F1: 0.9160
               precision    recall  f1-score   support

not clickbait       0.97      0.98      0.98       500
    clickbait       0.94      0.90      0.92       134

     accuracy                           0.97       634
    macro avg       0.95      0.94      0.95       634
 weighted avg       0.96      0.97      0.97       634



,Predicted 0,Predicted 1
Actual 0,492,8
Actual 1,14,120


## 7. Final test predictions and metrics

In [12]:
test_fusion = add_fusion_probability(scores["test"])
test_y = test_fusion[LABEL].to_numpy()
test_pred = test_fusion["final_prediction"].to_numpy()
test_metrics = pd.DataFrame([{
    "model": "soft_vote_text_ann_cnn",
    "test_rows": len(test_fusion),
    "test_accuracy": accuracy_score(test_y, test_pred),
    "test_f1": f1_score(test_y, test_pred, zero_division=0),
    "test_log_loss": log_loss(test_y, test_fusion["clickbait_probability"], labels=[0, 1]),
}])
prediction_path = OUTPUT_DIR / "test_fusion_predictions.csv"
test_fusion[[ID, *MODEL_COLUMNS, "clickbait_probability", "final_prediction"]].to_csv(prediction_path, index=False)
metrics_path = OUTPUT_DIR / "test_fusion_metrics.csv"
test_metrics.to_csv(metrics_path, index=False)
print("Saved predictions:", prediction_path)
print("Saved test metrics:", metrics_path)
display(test_metrics)
print(classification_report(test_y, test_pred, labels=[0, 1],
                            target_names=["not clickbait", "clickbait"], zero_division=0))

Saved predictions: d:\DC\multimodal-clickbait-detection-fresh\artifacts\fusion\predictions\test_fusion_predictions.csv
Saved test metrics: d:\DC\multimodal-clickbait-detection-fresh\artifacts\fusion\predictions\test_fusion_metrics.csv


,model,test_rows,test_accuracy,test_f1,test_log_loss
0,soft_vote_text_ann_cnn,634,0.949527,0.875,0.150354


               precision    recall  f1-score   support

not clickbait       0.96      0.98      0.97       500
    clickbait       0.92      0.84      0.88       134

     accuracy                           0.95       634
    macro avg       0.94      0.91      0.92       634
 weighted avg       0.95      0.95      0.95       634



## 7. Save fusion weights

The soft-voting fusion has three equal weights. Save them with the score order and decision threshold so deployment uses the exact same fusion rule.

In [ ]:
FUSION_WEIGHTS_PATH = ROOT / "models" / "fusion_weights.joblib"
fusion_artifact = {
    "model_columns": MODEL_COLUMNS,
    "weights": np.full(len(MODEL_COLUMNS), 1.0 / len(MODEL_COLUMNS), dtype=float),
    "threshold": 0.5,
    "method": "soft_vote_weighted_mean",
}
joblib.dump(fusion_artifact, FUSION_WEIGHTS_PATH)
print("Saved fusion weights:", FUSION_WEIGHTS_PATH)
